## years of life lost 
from https://genesis.destatis.de/datenbank/online/table/12621-0002/table-toolbar (year 2019) and https://service.destatis.de/bevoelkerungspyramide/#!y=2019

Corresponding Tables:
1) GENESIS 12621-0001 (period life table, 0002 is the years left) (12621-0001_de.csv)
2) Destatis nbr_of_people_of_age_file_file file (16_bevoelkerungsvorausberechnung_daten.csv)'

In [1]:
import json
import re

import numpy as np
import pandas as pd

For every age group a including person p with age x and sex s:
P(x,s| death in group a) ~ number_of_people_of_this_age_and_sex_{x,s} * mortality q_{x,s}.

In [2]:
prob_to_die_file = "12621-0001_de.csv"
nbr_of_people_of_age_file = "16_bevoelkerungsvorausberechnung_daten.csv"
period, year = "2019/21", 2019
age_groups = {"0-4": (0, 4), "5-15": (5, 15), "16-34": (16, 34), "35-59": (35, 59), "60-79": (60, 79), "80+": (80, 100)}


In [34]:
def column(sex, sex_row, var, var_row):  
    return next(j for j, (s, v) in enumerate(zip(sex_row, var_row)) if s == sex and var in v)

def read_prob_to_die_file(path, period):
    """Returns {'m': DataFrame(q, e), 'f': DataFrame(q, e)} indexed by age 0..100."""
    lines = open(path, encoding="utf-8-sig").read().splitlines()
    sex_row = next(l for l in lines if "männlich" in l).split(";")
    var_row = next(l for l in lines if "[q(x)]" in l).split(";")
    rows = [l.split(";") for l in lines if re.match(r"\d{4}/\d{2};", l)]
    rows = [r for r in rows if r[0] == period]

    age = [int(re.match(r"\d+", r[1]).group()) for r in rows]
    out = {}
    for key, sex in (("m", "männlich"), ("f", "weiblich")):
        # q ist die Sterbewahrscheinlichkeit und e sind die noch zu lebenden Jahre
        q, e = column(sex,  sex_row, "[q(x)]", var_row), column(sex,sex_row, "[e(x)]", var_row)
        out[key] = pd.DataFrame({"q": [float(r[q].replace(",", ".")) for r in rows],
                                 "e": [float(r[e].replace(",", ".")) for r in rows]},
                                index=age).sort_index()
    return out


def read_nbr_of_people_of_age_file(path, year):
    """nbr_of_people_of_age_file per sex as array over ages 0-100 (100 = open group 100+)."""
    df = pd.read_csv(path, sep=";", decimal=",")
    df = df[(df["Variante"] == 0) & (df["Simulationsjahr"] == year)]
    cols = [f"Bev_{x}_{x + 1}" for x in range(100)]
    return {("m" if r["mw"] == "m" else "f"):
            np.append(r[cols].to_numpy(float), r["Bev"] - r[cols].sum())
            for _, r in df.iterrows()}

In [ ]:
# atm: calculates women and men together
def yll_distributions(life, pop):
    rows = []
    for sex, table in life.items():
        e = table["e"].to_numpy()
        yll = np.append((e[:-1] + e[1:]) / 2, e[-1])     # death at age x + 0.5 on average and taking the years that are left
        # mulitply every sex & age with corresponding dying prob
        rows.append(pd.DataFrame({"age": table.index, "yll": yll, "w": pop[sex][table.index] * table["q"].to_numpy()})) 
    df = pd.concat(rows)

    out = {}
    for g, (low, high) in age_groups.items():
        #print("g", g)
        d = df[df["age"].between(low, high)]
        #print(d)
        prob = (d["w"] / d["w"].sum()).to_numpy()
        #print(prob)
        out[g] = {"yll": d["yll"].round(3).tolist(), "prob": prob.tolist(),
                  "mean_yll": float(prob @ d["yll"].to_numpy())}
    return out



In [50]:
dists = yll_distributions(read_prob_to_die_file(prob_to_die_file, period),
                            read_nbr_of_people_of_age_file(nbr_of_people_of_age_file, year))
print(dists)
with open("yll_2020.json", "w") as f:
    json.dump(dists, f)
for g, d in dists.items():
    print(f"{g:>6}: mean YLL per death = {d['mean_yll']:.2f}")

     age     yll         w
0      0  78.170  1.316580
1      1  77.305  0.092841
2      2  76.315  0.050614
3      3  75.325  0.046683
4      4  74.335  0.044071
..   ...     ...       ...
96    96   2.540  8.373726
97    97   2.375  6.631099
98    98   2.235  5.222186
99    99   2.100  3.412727
100  100   2.030  5.834472

[202 rows x 3 columns]
{'0-4': {'yll': [78.17, 77.305, 76.315, 75.325, 74.335, 83.0, 82.13, 81.145, 80.15, 79.155], 'prob': [0.46635518772815066, 0.03288579050605375, 0.017928416339411078, 0.016535790464787453, 0.015610794952452748, 0.3797793369714998, 0.027124420652656318, 0.01660463256276716, 0.014415312160162987, 0.01276031766205817], 'mean_yll': 80.03364177259303}, '5-15': {'yll': [73.345, 72.35, 71.355, 70.36, 69.365, 68.37, 67.375, 66.38, 65.385, 64.395, 63.405, 78.165, 77.17, 76.175, 75.18, 74.18, 73.185, 72.19, 71.195, 70.2, 69.205, 68.215], 'prob': [0.058039252507657416, 0.04844894487377561, 0.04984877229227191, 0.037857185734441794, 0.04136948981465908, 0.0